# 🏆 Capstone: End-to-End ML Project
## Real-World House Price Prediction

---

## 🎯 Project Goal
Predict house prices using the California Housing dataset.
We will apply **every concept** from this course:

```
✅ EDA
✅ Feature Engineering  
✅ Model Selection
✅ Cross-Validation
✅ Hyperparameter Tuning
✅ Final Evaluation
```

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
import warnings; warnings.filterwarnings('ignore')

print('Step 1/6: Libraries loaded ✅')

### Step 2: Load & Understand the Data

In [ ]:
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['Price'] = housing.target  # Price in $100,000

print(f'Shape: {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
print()
print(df.describe().round(2))

### Step 3: Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Price distribution
axes[0].hist(df['Price'], bins=50, color='steelblue', edgecolor='black')
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price ($100k)')

# Correlation with price
corr = df.corr()['Price'].drop('Price').sort_values()
colors = ['red' if c < 0 else 'green' for c in corr]
axes[1].barh(corr.index, corr.values, color=colors, edgecolor='black')
axes[1].set_title('Feature Correlation with Price')
axes[1].axvline(0, color='black')

# Scatter: Income vs Price
axes[2].scatter(df['MedInc'], df['Price'], alpha=0.1, s=5)
axes[2].set_title('Income vs Price')
axes[2].set_xlabel('Median Income')
axes[2].set_ylabel('Price')

plt.tight_layout()
plt.show()

### Step 4: Feature Engineering

In [ ]:
# Create new features
df['rooms_per_person']    = df['AveRooms'] / df['AveOccup']
df['bedrooms_per_room']   = df['AveBedrms'] / df['AveRooms']
df['income_per_room']     = df['MedInc'] / df['AveRooms']

print('New features created:')
print('  rooms_per_person  = AveRooms / AveOccup')
print('  bedrooms_per_room = AveBedrms / AveRooms')
print('  income_per_room   = MedInc / AveRooms')
print(f'\nDataset shape: {df.shape}  (added 3 new features)')

### Step 5: Train & Compare Models

In [ ]:
X = df.drop('Price', axis=1)
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    'Linear Regression': Pipeline([('sc', StandardScaler()), ('m', LinearRegression())]),
    'Ridge': Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=1.0))]),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

print(f'{"Model":<25} {"CV R²":<12} {"Test RMSE"}')
print('-' * 50)

best_model, best_score = None, -np.inf
for name, m in models.items():
    cv_r2 = cross_val_score(m, X_train, y_train, cv=5, scoring='r2').mean()
    m.fit(X_train, y_train)
    rmse = np.sqrt(mean_squared_error(y_test, m.predict(X_test)))
    print(f'{name:<25} {cv_r2:<12.4f} {rmse:.4f}')
    if cv_r2 > best_score:
        best_score, best_model = cv_r2, (name, m)

### Step 6: Final Evaluation

In [ ]:
name, model = best_model
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'🏆 Best Model: {name}')
print(f'   Test R²:   {r2:.4f}')
print(f'   Test RMSE: {rmse:.4f} (units: $100k)')
print(f'   Avg error: ${rmse * 100_000:,.0f}')

# Predicted vs Actual
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred, alpha=0.2, s=10)
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
plt.title(f'{name}\nR² = {r2:.4f}')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.legend()
plt.tight_layout()
plt.show()

### 🏋️ Your Challenge!

Can you beat the best model? Try:
1. Add more engineered features
2. Use `GridSearchCV` to tune the winning model
3. Try removing outliers (prices capped at 5.0)

In [ ]:
# ✏️ YOUR TURN: Can you improve R²?
# Hint: Remove capped prices (5.0) — they bias the model
df_clean = df[df['Price'] < 5.0].copy()
print(f'Rows removed (capped prices): {len(df) - len(df_clean)}')

X_c = df_clean.drop('Price', axis=1)
y_c = df_clean['Price']
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

rf_clean = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_clean.fit(X_tr, y_tr)
r2_clean = r2_score(y_te, rf_clean.predict(X_te))
print(f'R² after removing capped prices: {r2_clean:.4f}')
print(f'Improvement: +{(r2_clean - r2)*100:.1f}%')

## 🎓 You've Completed the ML Course!

### What You've Learned:

| Module | Concept |
|--------|--------|
| 01 | ML intro, bias-variance, workflow |
| 02 | Linear Regression, regularization |
| 03 | Logistic Regression, ROC-AUC |
| 04 | Decision Trees, Random Forests |
| 05 | SVM, kernel trick |
| 06 | KNN, Naive Bayes |
| 07 | K-Means clustering |
| 08 | PCA dimensionality reduction |
| 09 | Model evaluation, cross-validation |
| 10 | Feature engineering |
| 11 | AdaBoost, Gradient Boosting |
| 12 | Neural Networks |
| 13 | Hyperparameter tuning |
| 14 | End-to-end ML project |

### 🚀 Next Steps
- Deep Learning with PyTorch or TensorFlow
- Natural Language Processing (NLP)
- Computer Vision (CNN)
- MLOps — deploying models to production